# KilnWatch BD — YOLOv8 OBB Training

**Hack for Humanity Bangladesh 2026**

Train YOLOv8 oriented bounding-box (OBB) detectors on the
SentinelKilnDB dataset (CC BY-NC 4.0) converted to spatial-block splits.
Classes: **FCBK** (0) and **Zigzag** (1).

### Training plan
| Stage | Model | Image size | Epochs | Early stopping |
|-------|-------|-----------|--------|----------------|
| 0 | yolov8n-obb (smoke test) | 256 | 3 | – |
| 1 | yolov8n-obb | 256 | 50 | patience=10 |
| 1 | yolov8n-obb | 384 | 50 | patience=10 |
| 1 | yolov8n-obb | 512 | 50 | patience=10 |
| 2 | yolov8s-obb | best image size from stage 1 | 50 | patience=10 |

Chips are 128 × 128 px, so every image size above upsamples them (2×, 3×, 4×).

Augmentation on top of the Ultralytics defaults: `flipud=0.5`, `degrees=90`
(satellite imagery has no canonical orientation), `seed=0`.

### Selection protocol
1. Every run is scored on the **val** split (`model.val(split="val")`).
2. The best run is the one with the highest **val mAP50**.
3. Only that run's `best.pt` is evaluated on the **test** split, **once**. Test
   numbers are never used to choose anything.

> **⚠️ Licence:** Ultralytics is AGPL-3.0. Check implications before publishing.

## 0. Setup

In [ ]:
# Install only if Ultralytics is absent or older than the supported minimum.
from importlib.metadata import PackageNotFoundError, version

from packaging.version import Version

try:
    installed_ultralytics = Version(version("ultralytics"))
except PackageNotFoundError:
    installed_ultralytics = Version("0")

if installed_ultralytics < Version("8.1"):
    %pip install -q "ultralytics>=8.1"
else:
    print(f"Ultralytics {installed_ultralytics} already meets the minimum; skipping pip install.")

In [ ]:
import gc
import json
import math
import shutil
from pathlib import Path

import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("Select a Colab/Kaggle GPU runtime before training.")

## 1. Mount Drive & set paths

Adjust `DATASET_ROOT` / `OUTPUT_DIR` to where you placed the converted dataset
(output of `kilnwatch convert`). The local defaults come first; uncomment one of
the Colab / Kaggle blocks below them to override.

Use a dataset converted **after** the 1300 m Chebyshev leakage filter change
(section 2 warns if `split_report.json` predates it).

In [ ]:
# --- Local (for testing only) ---
DATASET_ROOT = Path("data/interim/yolo_obb")
OUTPUT_DIR = Path("runs")

# --- Google Colab (uncomment to override) ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATASET_ROOT = Path("/content/drive/MyDrive/kilnwatch/yolo_obb")
# OUTPUT_DIR = Path("/content/drive/MyDrive/kilnwatch/runs")

# --- Kaggle (uncomment to override) ---
# DATASET_ROOT = Path("/kaggle/input/kilnwatch-yolo-obb")
# OUTPUT_DIR = Path("/kaggle/working/runs")

# Absolute paths: Ultralytics prefixes a relative `project` with its own runs_dir.
DATASET_ROOT = DATASET_ROOT.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()

DATASET_YAML = DATASET_ROOT / "dataset.yaml"
assert DATASET_YAML.exists(), f"dataset.yaml not found at {DATASET_YAML}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATASET_ROOT}")
print(f"Output:  {OUTPUT_DIR}")

## 2. Dataset sanity check

In [ ]:
# Check split sizes
for split in ["train", "val", "test"]:
    imgs = list((DATASET_ROOT / split / "images").glob("*.png"))
    lbls = list((DATASET_ROOT / split / "labels").glob("*.txt"))
    print(f"{split:>5}: {len(imgs)} images, {len(lbls)} labels")
    assert imgs, f"no images found for split '{split}'"

# Class distribution from the split report: {split: {class: n}}
report_path = DATASET_ROOT / "split_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    for split, per_class in report["counts_per_split_per_class"].items():
        print(f"\n{split}:")
        for cls, n in per_class.items():
            print(f"  {cls}: {n}")

    threshold_m = report.get("leakage_filter_threshold_m")
    if threshold_m is None:
        print(
            "\nWARNING: split_report.json has no leakage_filter_threshold_m, so this "
            "dataset was converted before the Chebyshev leakage filter. Re-run convert "
            "before trusting val/test numbers."
        )
    else:
        print(f"\nLeakage filter: {report['leakage_filter_metric']}, threshold {threshold_m} m")
else:
    print(f"\nWARNING: {report_path} not found; skipping class counts.")

## 3. Absolute dataset YAML

The converter writes `path: .` into `dataset.yaml`, which Ultralytics resolves
relative to wherever it is run. We write a copy with `path` set to the absolute
`DATASET_ROOT` and use **that** file for every train / val / test call.

In [ ]:
ABS_YAML = Path("/content/dataset_abs.yaml")

with open(DATASET_YAML) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(DATASET_ROOT)

names = {int(k): v for k, v in data_cfg["names"].items()}
assert names == {0: "FCBK", 1: "Zigzag"}, f"unexpected class names: {names}"
for key in ("train", "val", "test"):
    assert (DATASET_ROOT / data_cfg[key]).is_dir(), f"missing {key} dir: {data_cfg[key]}"

try:
    ABS_YAML.parent.mkdir(parents=True, exist_ok=True)
    ABS_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
except OSError:
    # /content only exists on Colab; fall back so local dry-runs still work.
    ABS_YAML = OUTPUT_DIR / "dataset_abs.yaml"
    ABS_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
    print("WARNING: /content is not writable here; using the fallback path below.")

print(f"Wrote {ABS_YAML}:\n")
print(ABS_YAML.read_text())

## 4. Training helpers

Notes on what these do (all checked against the Ultralytics source, not run):

* `model.train()` returns a metrics object, **not** a results object with an
  `epoch` field, so the number of epochs actually run is read from `results.csv`.
* Per-class arrays (`box.p`, `box.r`, `box.ap50`, `box.ap`) are ordered by
  `box.ap_class_index`, **not** by class id, so they are mapped through it.
  A class with no ground truth in the evaluated split is reported as `null`.
* Ultralytics picks each run's `best.pt` by its own fitness (mAP50-95 on val).
  Runs are then compared with each other by val mAP50, as specified.

In [ ]:
EPOCHS = 50
PATIENCE = 10
BATCH = 16  # lower this if yolov8s-obb at imgsz 512 runs out of GPU memory
AUG = {"flipud": 0.5, "degrees": 90, "seed": 0}

results_log = []   # one entry per training run
SMOKE_OK = False   # set True by the smoke-test cell
TEST_DONE = False  # set True after the single test-split evaluation


def _f(x):
    """float(x), or None if it is not finite (keeps the JSON strictly valid)."""
    x = float(x)
    return x if math.isfinite(x) else None


def fmt(x):
    return "n/a" if x is None else f"{x:.4f}"


def score(entry):
    """Selection score: val mAP50 (missing/NaN sorts last)."""
    v = entry["val"]["mAP50"]
    return -math.inf if v is None else v


def per_class_metrics(res):
    """Map per-class metric arrays to class names through ap_class_index."""
    names_map = {int(k): v for k, v in res.names.items()}
    out = {names_map[i]: None for i in sorted(names_map)}  # None = class absent from split
    nt = getattr(res, "nt_per_class", None)  # indexed by class id
    box = res.box
    for pos, cls_idx in enumerate(box.ap_class_index):
        cls_idx = int(cls_idx)
        out[names_map[cls_idx]] = {
            "precision": _f(box.p[pos]),
            "recall": _f(box.r[pos]),
            "mAP50": _f(box.ap50[pos]),
            "mAP50_95": _f(box.ap[pos]),
            "instances": int(nt[cls_idx]) if nt is not None and len(nt) > cls_idx else None,
        }
    return out


def summarize(res):
    """Overall + per-class metrics from a validator result."""
    return {
        "mAP50": _f(res.box.map50),
        "mAP50_95": _f(res.box.map),
        "precision": _f(res.box.mp),
        "recall": _f(res.box.mr),
        "per_class": per_class_metrics(res),
    }


def evaluate(weights_path, split, imgsz, run_name):
    """Run model.val on `split` with the absolute YAML. Returns (summary, save_dir)."""
    model = YOLO(str(weights_path))
    res = model.val(
        data=str(ABS_YAML),
        split=split,
        imgsz=imgsz,
        batch=BATCH,
        plots=True,
        project=str(OUTPUT_DIR),
        name=run_name,
        exist_ok=True,
        verbose=False,
    )
    save_dir = Path(getattr(res, "save_dir", OUTPUT_DIR / run_name))
    summary = summarize(res)
    del model
    gc.collect()
    return summary, save_dir


def epochs_from_results_csv(run_dir):
    """Epochs actually run = rows in results.csv (early stopping may cut it short)."""
    df = pd.read_csv(Path(run_dir) / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    return len(df)


PLOT_GLOBS = ("confusion_matrix*.png", "*PR_curve.png")  # prefix varies by version (BoxPR_curve.png)


def copy_plots(src_dir, dest_dir, prefix):
    """Copy the confusion matrices and PR curves from `src_dir` to `dest_dir`."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in PLOT_GLOBS:
        for p in sorted(Path(src_dir).glob(pattern)):
            dest = dest_dir / f"{prefix}_{p.name}"
            shutil.copy2(p, dest)
            copied.append(dest.name)
    return copied


def save_progress():
    """Write the log so far (so a dropped Colab session does not lose finished runs)."""
    with open(OUTPUT_DIR / "training_results.json", "w") as f:
        json.dump({"status": "in_progress", "runs": results_log}, f, indent=2)


def run_config(cfg, epochs=EPOCHS, patience=PATIENCE):
    """Train one config, then score its best.pt on the VAL split."""
    name = cfg["name"]
    print(f"\n{'=' * 60}\nTraining: {name} (epochs={epochs}, patience={patience})\n{'=' * 60}\n")

    model = YOLO(cfg["model"])
    model.train(
        data=str(ABS_YAML),
        epochs=epochs,
        imgsz=cfg["imgsz"],
        batch=BATCH,
        patience=patience,
        project=str(OUTPUT_DIR),
        name=name,
        exist_ok=True,
        save=True,
        plots=True,
        verbose=True,
        **AUG,
    )
    run_dir = Path(model.trainer.save_dir)  # the directory Ultralytics really used
    best_pt = run_dir / "weights" / "best.pt"
    assert best_pt.exists(), f"best.pt not found at {best_pt}"
    epochs_run = epochs_from_results_csv(run_dir)
    del model
    gc.collect()
    torch.cuda.empty_cache()

    val_summary, val_dir = evaluate(best_pt, "val", cfg["imgsz"], f"{name}_val")
    return {
        "name": name,
        "model": cfg["model"],
        "imgsz": cfg["imgsz"],
        "epochs_requested": epochs,
        "epochs_run": epochs_run,
        "run_dir": str(run_dir),
        "best_pt": str(best_pt),
        "val_dir": str(val_dir),
        "val": val_summary,
    }


def print_metrics(label, m):
    print(f"{label}: mAP50={fmt(m['mAP50'])}  mAP50-95={fmt(m['mAP50_95'])}  "
          f"P={fmt(m['precision'])}  R={fmt(m['recall'])}")
    for cls_name, c in m["per_class"].items():
        if c is None:
            print(f"   {cls_name}: no ground truth in this split")
        else:
            print(f"   {cls_name}: P={fmt(c['precision'])} R={fmt(c['recall'])} "
                  f"mAP50={fmt(c['mAP50'])} mAP50-95={fmt(c['mAP50_95'])} "
                  f"(instances={c['instances']})")


def print_entry(entry):
    print(f"\n✅ {entry['name']}: {entry['epochs_run']}/{entry['epochs_requested']} epochs")
    print_metrics("   VAL", entry["val"])

## 5. Smoke test (run this first)

Three epochs of `yolov8n-obb` at 256. It exercises the exact code path used by
the full runs (absolute YAML, augmentation arguments, `results.csv` parsing,
per-class mapping, plot collection, JSON serialisation) but **never touches the
test split**. The full-training cells refuse to run until this cell passes.

In [ ]:
SMOKE_OK = False

smoke_cfg = {"model": "yolov8n-obb.pt", "imgsz": 256, "name": "smoke-yolov8n-obb-256"}
smoke = run_config(smoke_cfg, epochs=3)
print_entry(smoke)

assert smoke["epochs_run"] == 3, f"expected 3 epochs in results.csv, got {smoke['epochs_run']}"
assert smoke["val"]["mAP50"] is not None, "val mAP50 is NaN/None"
assert set(smoke["val"]["per_class"]) == {"FCBK", "Zigzag"}, smoke["val"]["per_class"].keys()
json.dumps(smoke)  # must be JSON serialisable

copied = copy_plots(smoke["val_dir"], OUTPUT_DIR / "smoke_artifacts", "val")
print("Plots collected:", copied)
assert any("confusion_matrix" in c for c in copied), "no confusion matrix found"
assert any("PR_curve" in c for c in copied), "no PR curve found"

SMOKE_OK = True
print("\nSmoke test passed. Continue with the full runs.")

## 6. Training

**Stage 1:** `yolov8n-obb` at imgsz 256 / 384 / 512.
Each run uses pretrained COCO-OBB weights, early stopping and is scored on **val**.

In [ ]:
assert SMOKE_OK is True, "Run the smoke-test cell (section 5) first."

# yolov8s-obb is appended after stage 1, once the best image size is known.
CONFIGS = [
    {"model": "yolov8n-obb.pt", "imgsz": s, "name": f"yolov8n-obb-{s}"}
    for s in (256, 384, 512)
]

results_log.clear()
for cfg in CONFIGS:
    entry = run_config(cfg)
    results_log.append(entry)
    save_progress()
    print_entry(entry)

**Stage 2:** `yolov8s-obb` at the image size whose stage-1 run had the highest val mAP50.

In [ ]:
assert SMOKE_OK is True, "Run the smoke-test cell (section 5) first."
assert len(results_log) == 3, "Run stage 1 (all three yolov8n-obb configs) first."

best_n = max(results_log, key=score)
S_CFG = {
    "model": "yolov8s-obb.pt",
    "imgsz": best_n["imgsz"],
    "name": f"yolov8s-obb-{best_n['imgsz']}",
}
print(f"Best stage-1 image size: {best_n['imgsz']} (val mAP50={fmt(best_n['val']['mAP50'])})")

CONFIGS.append(S_CFG)
entry = run_config(S_CFG)
results_log.append(entry)
save_progress()
print_entry(entry)

## 7. Results comparison (val split)

In [ ]:
rows = []
for e in results_log:
    row = {
        "Config": e["name"],
        "Epochs run": f"{e['epochs_run']}/{e['epochs_requested']}",
        "val mAP50": e["val"]["mAP50"],
        "val mAP50-95": e["val"]["mAP50_95"],
    }
    for cls_name, c in e["val"]["per_class"].items():
        row[f"{cls_name} P"] = None if c is None else c["precision"]
        row[f"{cls_name} R"] = None if c is None else c["recall"]
        row[f"{cls_name} mAP50"] = None if c is None else c["mAP50"]
    rows.append(row)

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("VALIDATION RESULTS (selection split)")
print("=" * 80)
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 8. Select the best run (val mAP50) and evaluate it on test — once

The test split is scored a single time, on the run chosen above. The cell
refuses to run a second time in the same session.

In [ ]:
assert TEST_DONE is False, "The test split was already evaluated; refusing to look at it again."
assert len(results_log) == 4, "Run both training stages first."

best = max(results_log, key=score)
assert score(best) > -math.inf, "No run has a finite val mAP50."
print(f"Selected by val mAP50: {best['name']} (val mAP50={fmt(best['val']['mAP50'])})")

test_summary, test_dir = evaluate(best["best_pt"], "test", best["imgsz"], "test_eval")
TEST_DONE = True

print(f"\nTEST split, single evaluation of {best['name']}:")
print_metrics("   TEST", test_summary)

## 9. Save artifacts

Copies `best.pt`, the val + test confusion matrices and PR curves, and writes
`training_results.json` to `OUTPUT_DIR`.

In [ ]:
# best.pt of the selected run
dest_pt = OUTPUT_DIR / "best.pt"
shutil.copy2(best["best_pt"], dest_pt)
print(f"Saved: {dest_pt}")

# Confusion matrices + PR curves (val plots from the selected run, test plots from the one test eval)
val_plots = copy_plots(best["val_dir"], OUTPUT_DIR, "val")
test_plots = copy_plots(test_dir, OUTPUT_DIR, "test")
missing = [label for label, found in (("val", val_plots), ("test", test_plots)) if not found]
if missing:
    print(f"WARNING: no confusion matrix / PR curve PNGs found for: {missing}")
print("Plots:", val_plots + test_plots)

payload = {
    "status": "complete",
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "dataset_root": str(DATASET_ROOT),
    "classes": [names[i] for i in sorted(names)],
    "hyperparameters": {"epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH, **AUG},
    "selection": {
        "split": "val",
        "metric": "mAP50",
        "selected_run": best["name"],
        "stage2_imgsz": S_CFG["imgsz"],
    },
    "runs": results_log,
    "test": {
        "run": best["name"],
        "imgsz": best["imgsz"],
        "evaluated_once": True,
        "save_dir": str(test_dir),
        **test_summary,
    },
    "artifacts": {
        "best_pt": str(dest_pt),
        "val_plots": val_plots,
        "test_plots": test_plots,
    },
}
log_path = OUTPUT_DIR / "training_results.json"
with open(log_path, "w") as f:
    json.dump(payload, f, indent=2, allow_nan=False)
print(f"Results log saved to {log_path}")
print("\n📋 Paste training_results.json back to the agent for documentation.")

---

## Next steps

1. Copy `best.pt` to your project
2. Run `kilnwatch infer best.pt` to detect kilns in exported Sentinel-2 composites
3. Run `kilnwatch check-rules` → `kilnwatch score`
4. Launch the dashboard: `streamlit run app/streamlit_app.py`